In [1]:
import json
import pandas as pd

# Load your JSON file with UTF-8 encoding
with open('graph.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Load your CSV metadata file
metadata_df = pd.read_csv('data/Mastodon_instances_metadata_v3.csv')
metadata_df = metadata_df.set_index('instance')
metadata_df = metadata_df.where(pd.notnull(metadata_df), None)  # Replace NaN with None

# Prepare new list of nodes with metadata
new_nodes = []
for node in data['nodes']:
    node_id = node['id']
    if node_id in metadata_df.index:
        meta = metadata_df.loc[node_id].to_dict()
        # Remove keys with None values
        meta = {k: v for k, v in meta.items() if v is not None}
        node['metadata'] = meta
        new_nodes.append(node)

# Replace nodes with filtered/augmented list
data['nodes'] = new_nodes

# Save the updated JSON with UTF-8 encoding
with open('graph_with_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False) 

In [ ]:
# Add similarities as nested attributes to edges with dynamic control
import json

# Set to None to include all available similarities automatically
SIMILARITY_TYPES = None

# Loading the graph with metadata
with open('graph_with_metadata.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Loading the similarities
with open('similarities.json', 'r', encoding='utf-8') as f:
    similarities = json.load(f)

# Build a lookup for (source, target) -> similarity dict
sim_lookup = {}
for entry in similarities:
    key = (entry['source'], entry['target'])
    sim_lookup[key] = entry

# Add similarities to edges in nested structure
for edge in data.get('edges', []):
    source = edge.get('source')
    target = edge.get('target')
    sim = sim_lookup.get((source, target))
    if sim:
        edge_similarities = {}
        
        # If SIMILARITY_TYPES is None, include all available similarities
        if SIMILARITY_TYPES is None:
            available_similarities = [k for k in sim.keys() if k not in ('source', 'target')]
        else:
            available_similarities = SIMILARITY_TYPES
        
        # Add only the specified similarity types
        for sim_type in available_similarities:
            if sim_type in sim:
                edge_similarities[sim_type] = sim[sim_type]
        
        # Only add similarities key if we have similarities to add
        if edge_similarities:
            edge['similarities'] = edge_similarities

# Save the updated graph JSON with edge similarities
with open('graph_similarities_with_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Similarities added to edges and saved. Included similarity types: {SIMILARITY_TYPES or 'all available'}")

Similarities added to edges and saved. Included similarity types: all available


In [ ]:
# Cleaning the graph
with open('graph_similarities_with_metadata.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

node_ids = set(node['id'] for node in data.get('nodes', []))

clean_edges = [
    edge for edge in data.get('edges', [])
    if edge.get('source') in node_ids and edge.get('target') in node_ids
]

removed = len(data.get('edges', [])) - len(clean_edges)
print(f"Original edges: {len(data.get('edges', []))}, Cleaned edges: {len(clean_edges)}, Removed: {removed}")

data['edges'] = clean_edges

with open('graph_similarities_with_metadata_cleaned.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2) 

Original edges: 129234, Cleaned edges: 117243, Removed: 11991
